# Scale AI Interview Practice
## ML Coding — LLM Practical

---

## Part 1: Data Pipeline

You have `data/verdict.txt` — a raw text file.

Your model's `nn.Embedding` layer needs **integer indices**, not strings.

**Your tasks:**
1. Load `verdict.txt`
2. Build a `vocab` — a mapping of `word → index`
3. Write `text_to_indices(text, vocab)` to convert the full text into a list of integers

**Bonus (senior touch):** How do you handle a word that isn't in your vocabulary? Think `<UNK>` token.

**Before you code — answer this first:**
> What data structure do you reach for to build the vocab, and what is the very first thing you do to the raw text before building it?

In [8]:
import re

# Part 1: Load verdict.txt
with open("../data/verdict.txt" , "r", encoding="utf-8") as f:
    verdict = f.read()

In [4]:
# Part 1: Build vocab (word -> index mapping)
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text) 
    return text

def build_vocab(text):

    # Clean the text
    text = clean_text(text)
    words = sorted(set(text.split()))
    vocab = {word:idx for idx,word in enumerate(words)}
    vocab["<UNK>"] = len(vocab)
    vocab["<|endoftext|>"] = len(vocab) 
    return vocab


In [5]:
# Part 1: text_to_indices(text, vocab)
def text_to_indices(txt, vocab):
    txt = clean_text(txt)
    words = txt.split()
    indices = [vocab.get(word, vocab["<UNK>"]) for word in words]
    indices.append(vocab["<|endoftext|>"])
    return indices

---

## Part 2: Debugging — NaN Loss (Attention Scaling)

You are given this buggy attention function and told the loss is NaN:

```python
def buggy_attention_scores(q, k):
    # q: [Batch, Heads, Seq, Dim]
    # k: [Batch, Heads, Seq, Dim]
    scores = q @ k.transpose(-2, -1)
    attn_weights = torch.softmax(scores, dim=-1)
    return attn_weights
```

**Q1:** `q` and `k` are already `[Batch, Heads, Seq, Dim]` — `.view()` and `.transpose(1,2)` were done upstream. When you do `q @ k.transpose(-2, -1)`, what is the shape of `scores`, and why does the magnitude grow with `Dim`?

**Q2:** If `scores` contains a value like `100.0`, what does `exp(100)` evaluate to, and what happens in PyTorch when float32 can't represent that number?

**Q3:** What is the one-line fix, and why divide by `sqrt(k.shape[-1])` specifically?

---

**Answer:**
> The reshape/transpose is already done upstream so `(-2, -1)` is correct for the scores matmul.
> The bug is missing the scaling: dot products grow in magnitude with `Dim` because each element is a sum of `Dim` multiplications.
> Without scaling, large values go into softmax → `exp(100)` overflows float32 → `inf` → `NaN`.
> Fix: divide scores by `sqrt(k.shape[-1])` to normalize the variance of the dot product back to ~1.

In [ ]:
import torch

# Fixed attention scores
def attention_scores(q, k):
    # q, k: [Batch, Heads, Seq, Dim]
    scores = q @ k.transpose(-2, -1)      # [Batch, Heads, Seq, Seq]
    scores = scores / k.shape[-1] ** 0.5  # scale to prevent overflow
    attn_weights = torch.softmax(scores, dim=-1)
    return attn_weights


### Part 3 — Sliding Window (Input/Target Pairs)                                                                                                                                    
                                                                                                                                                                                  
You have this from Part 1:                                                                                                                                                      
                                                                                                                                                                                
indices = [4, 17, 203, 45, 8, 99, 12, 7, ...]  # full text as token ids                                                                                                         
                                                                                                                                                                                
Your model is trained to predict the next token. So for every input sequence, the target is the same sequence shifted by 1.                                                     
                                                                                                                                                                                
With context_length = 4:                                                                                                                                                        
                
input:  [4,  17, 203, 45]                                                                                                                                                       
target: [17, 203, 45,  8]
                                                                                                                                                                                
input:  [17, 203, 45,  8]
target: [203, 45,  8, 99]                                                                                                                                                       
...                                                                                                                                                                             

Your tasks:                                                                                                                                                                     
                
- Write create_sliding_window_pairs(indices, context_length, stride) that returns a list of (input, target) tuples as PyTorch LongTensors                                      
- What does stride control? What happens when stride=1 vs stride=context_length?
                                                                                                                                                                                
Before coding — one question:                                                                                                                                                   
▎ If your text has N total tokens and context_length = 4, stride = 1, how many pairs do you get? Write the formula.      

In [ ]:
def create_sliding_window_pairs(indices, context_length, stride):
    pairs = []
    for i in range(0, len(indices) - context_length , stride):
        input_seq = indices[i:i+context_length]
        target_seq = indices[i+1:i+1+context_length]
        pairs.append((input_seq, target_seq))
    return pairs


#### Answer:

Stride explanation — good, but sharpen it for the interview: 

    stride=1 → maximum overlap, more pairs, slower training, risk of overfitting on small datasets
    stride=context_length → no overlap, fewest pairs, faster but misses cross-boundary patterns  

 #### Next question: now wrap this in a PyTorch Dataset and DataLoader.
                                                                                                                                                                                  
from torch.utils.data import Dataset, DataLoader
                                                                                                                                                                                
Write a TextDataset class that takes indices, context_length, stride and implement __len__ and __getitem__. Then create a DataLoader from it with batch_size=2.                 

What shape does one batch have?  

In [12]:
from torch.utils.data import DataLoader, Dataset

class SimpleTextDatset(Dataset):
    def __init__(self, indices, context_length, stride):
        self.pairs = []

        for i in range(0, len(indices) - context_length + 1, stride):
            input_chunk = indices[i:i+context_length]
            target_chunk = indices[i+1:i+1+context_length]
            self.pairs.append((torch.tensor(input_chunk), torch.tensor(target_chunk)))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]
    
def create_dataloader(batch, indices, context_length, stride):
    dataset = SimpleTextDatset(indices, context_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch, shuffle=False)
    return dataloader

### End-to-end Pipeline

In [15]:
import re
import torch
from torch.utils.data import DataLoader, Dataset

class TextDataset(Dataset):
    def __init__(self, indices, context_length, stride):
        self.indices = indices
        self.context_length = context_length
        self.stride = stride

    def __len__(self):
        return (len(self.indices) - self.context_length + 1 ) // self.stride
    
    def __getitem__(self, idx):
        start_idx = idx * self.stride
        input_seq = self.indices[start_idx : start_idx + self.context_length]
        target_seq = self.indices[start_idx + 1 : start_idx + 1 + self.context_length]
        return torch.tensor(input_seq), torch.tensor(target_seq)

class DataLoaderV1:
    def __init__(self, datapath, batch_size, context_length, stride):

        with open(datapath, 'r') as file:
            self.text = file.read()

        self.context_length = context_length
        self.batch_size = batch_size
        self.stride = stride
        self.vocab = self.build_vocab()
        self.indices = self.text_to_indices()
        self.dataset = TextDataset(self.indices, self.context_length, self.stride)
        self.dataloader = DataLoader(self.dataset, batch_size=self.batch_size, shuffle=True)

    def clean_text(self, text):
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text) 
        return text
    
    def build_vocab(self):

        # Clean the text
        text = self.clean_text(self.text)
        words = sorted(set(text.split()))
        vocab = {word:idx for idx,word in enumerate(words)}
        vocab["<UNK>"] = len(vocab)
        vocab["<|endoftext|>"] = len(vocab) 
        return vocab
    
    def text_to_indices(self):
        txt = self.clean_text(self.text)
        words = txt.split()
        indices = [self.vocab.get(word, self.vocab["<UNK>"]) for word in words]
        indices.append(self.vocab["<|endoftext|>"])
        return indices
    

dl = DataLoaderV1("../data/verdict.txt", batch_size=3, context_length=4, stride=1)                                                                                              
                
input_batch, target_batch = next(iter(dl.dataloader))                                                                                                                           
print("Input shape:", input_batch.shape)   # [3, 4]
print("Target shape:", target_batch.shape) # [3, 4]                                                                                                                             
print("Input batch:\n", input_batch)                                                                                                                                            
print("Target batch:\n", target_batch)


Input shape: torch.Size([3, 4])
Target shape: torch.Size([3, 4])
Input batch:
 tensor([[ 317,  983,  985,    8],
        [1062,  463,  563,  893],
        [ 123,  488,  359,   15]])
Target batch:
 tensor([[ 983,  985,    8,  285],
        [ 463,  563,  893, 1124],
        [ 488,  359,   15,  171]])


In [16]:
import json

with open("../data/data.json", "r" , encoding="utf-8") as f:
    data = json.load(f)

print(data.keys())
text = " ".join(data["sentences"])
print(len(text))

dict_keys(['dataset', 'description', 'sentences'])
610


### Tokenizer :Pre-built

In [18]:
import tiktoken

encoding = tiktoken.get_encoding("gpt2")
text = "Hello world! This is a test."
tokens = encoding.encode(text)
print(tokens)

print(encoding.n_vocab)

[15496, 995, 0, 770, 318, 257, 1332, 13]
50257


### Embeddings

Question 1 — Conceptual:                                                                                                                                                               
                                                                                                                                                                                         
    You have vocab_size = 1000 and you want each token represented as a 64-dimensional vector. Write the one line to create the embedding layer and explain what's happening inside it.
 

#### Q3 was:                                                                                                                                                                                
                                                                                                                                                                                         
    Your embedding only knows what each token is, not where it is in the sequence. Two sentences "cat sat" and "sat cat" would produce identical embeddings. How do you fix this?

In [31]:
import torch                                                                                                                                     
import torch.nn as nn                                                                                                                            
                                                                                                                                                
batch_size = 3                                                                                                                                   
context_length = 4                                                                                                                               
emb_dim = 64
vocab_size = 1000                                                                                                                                
                                        
# Token + Positional embeddings
token_emb = nn.Embedding(vocab_size, emb_dim)                                                                                                    
pos_emb   = nn.Embedding(context_length, emb_dim)
                                                                                                                                                
# Input         
input_batch = torch.randint(0, vocab_size, (batch_size, context_length))  # [3, 4]                                                               
positions   = torch.arange(context_length)      

final_emb = token_emb(input_batch) + pos_emb(positions)  # [3, 4, 64]
print(final_emb.shape)  # [3, 4, 64]

torch.Size([3, 4, 64])


#### Causal Masking:                                                                                                                  
                                 
After computing scores = Q @ K.T / sqrt(d) shape [B, H, T, T] — why do we mask the upper triangle with -inf before softmax, and what happens to  
those -inf values after softmax?            
                                        
Answer that and we continue down the chain.

In [36]:
# LLMs are autoregressive — when predicting token at position t, it should only attend to positions 0 to t. Future tokens don't exist yet at 
# inference time, so we mask them during training too for consistency. 

# Why -inf specifically:                                                                                                                           
# Because softmax(exp(x)) — exp(-inf) = 0, so those positions get exactly zero attention weight. Any other large negative would get close to zero
# but not exactly.                                                                                                                                
                                                                                                                                                   
# How it's applied:                                                                                                                                
# scores = Queries @ Keys.transpose(-2, -1)  # [B, H, T, T]                                                                                     
                                                                                                                                                
# mask = torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()                                                                                           
# scores = scores.masked_fill(mask, float('-inf'))                                                                                                 
                                                  
# weights = torch.softmax(scores / Keys.shape[-1]**0.5, dim=-1)  # -inf → 0                                                                                              

In [39]:
class TransformerBlock(nn.Module):                                                                                                               
    def __init__(self, cfg):                                                                                                                     
        super().__init__()                                                                                                                       
        self.att = MultiHeadAttention(                                                                                                           
            d_in=cfg["emb_dim"],                                                                                                                 
            d_out=cfg["emb_dim"],           
            num_heads=cfg["n_heads"],                                                                                                            
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"]                                                                                                             
        )                               
        self.ffn = FeedForward(cfg)                                                                                                              
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])                                                                                                
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.dropout = nn.Dropout(cfg["drop_rate"])                                                                                              
                                            
    def forward(self, x):                                                                                                                        
        # Skip + Attention
        x = x + self.dropout(self.att(self.norm1(x)))                                                                                            
        # Skip + FFN                        
        x = x + self.dropout(self.ffn(self.norm2(x)))                                                                                            
        return x
                                                                                                                                                
                                            
class GPTModel(nn.Module):                                                                                                                       
    def __init__(self, cfg):
        super().__init__()                                                                                                                       
        self.token_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb   = nn.Embedding(cfg["context_length"], cfg["emb_dim"])                                                                     
        self.dropout   = nn.Dropout(cfg["drop_rate"])
        self.blocks    = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.norm      = nn.LayerNorm(cfg["emb_dim"])                                                                                            
        self.out_head  = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
                                                                                                                                                
    def forward(self, x):                   
        B, T = x.shape                                                                                                                           
        x = self.token_emb(x) + self.pos_emb(torch.arange(T))  # [B, T, emb_dim]
        x = self.dropout(x)                                                                                                                      
        x = self.blocks(x)                                       # [B, T, emb_dim]
        x = self.norm(x)                                         # [B, T, emb_dim]                                                               
        logits = self.out_head(x)                                # [B, T, vocab_size]                                                            
        return logits                       

In [41]:
# Q1 — Buggy sampler, find the bug:                                                                                                                
                                                                                                                                                   
def sample_next_token(logits, k=5):                                                                                                              
    # logits: [vocab_size]                                                                                                                       
    top_k_values, top_k_indices = torch.topk(logits, k)
    probs = torch.softmax(top_k_values, dim=-1)                                                                                                  
    next_token = top_k_indices[torch.argmax(probs)]
    return next_token  


# the real bug — torch.argmax always picks the highest probability token — that's greedy sampling, not random sampling. So it always
# returns the same token.                                                                                                                         

# Fix — use torch.multinomial instead:                                                                                                             
                                                                                                                                                   
def sample_next_token(logits, k=5):
    # logits: [vocab_size]                                                                                                                       
    top_k_values, top_k_indices = torch.topk(logits, k)
    probs = torch.softmax(top_k_values, dim=-1)                                                                                                  
    sampled = torch.multinomial(probs, num_samples=1)  # random sample
    next_token = top_k_indices[sampled]     
    return next_token                                                                                                                            
   
# torch.multinomial samples proportional to probability — token with 0.5 prob gets picked 50% of the time, not always. 

In [42]:
import json                                                                                                                                      
import csv                                                                                                                                       
                                                                                                                                                
# Generate synthetic training pairs                                                                                                              
data = [{"input": "the cat", "target": "sat"},
        {"input": "the dog", "target": "runs"}]                                                                                                  
                                                                            
# Save as JSON                                                                
with open("data.json", "w") as f:     
    json.dump(data, f)           
                                                                                                                                                
# Save as CSV                                                                 
with open("data.csv", "w") as f:                                                                                                                 
    writer = csv.DictWriter(f, fieldnames=["input", "target"])
    writer.writeheader()                                                                                                                         
    writer.writerows(data)    

In [43]:
# Training loop — buggy code, find the bugs:                                                                                                       
                                                                                                                                                   
model = GPTModel(cfg)                                                                                                                            
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)                                                                                      
                                                                                                                                                
for epoch in range(10):                                                                                                                          
    for input_batch, target_batch in dataloader:
                                                                                                                                                
        logits = model(input_batch)                                                                                                              
        loss = nn.CrossEntropyLoss()(logits.view(-1, vocab_size), target_batch.view(-1))                                                         
                                                                                                                                                
        loss.backward()                                                       
        optimizer.step()                                                                                                                         

        print(f"Loss: {loss.item()}")   


# Fixed one

model.train()                                                                                                                                    
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)                                                                                      
                                                            
for epoch in range(10):
    for input_batch, target_batch in dataloader:                                                                                                 
        optimizer.zero_grad()                   
        logits = model(input_batch)                                                                                                              
        loss = nn.CrossEntropyLoss()(logits.view(-1, vocab_size), target_batch.view(-1))
        loss.backward()                                                                 
        optimizer.step()                                                     
        print(f"Loss: {loss.item()}")               

NameError: name 'cfg' is not defined